In [1]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os
import requests
PASSWORD = os.getenv('SNOWSQL_PWD')
print(PASSWORD)

5eWxWv4EvyCkhkY


In [3]:
try:
    ctx = snowflake.connector.connect(
        user='ABHINAVSHARMA2002',
        password=PASSWORD,
        account='qraojwa-yg67137'
    )
    cs = ctx.cursor()
    try:
        cs.execute("CREATE WAREHOUSE IF NOT EXISTS task_1_warehouse_mg")
        cs.execute("CREATE DATABASE IF NOT EXISTS testdb_mg")
        cs.execute("USE DATABASE testdb_mg")
        cs.execute("CREATE SCHEMA IF NOT EXISTS task_2_mg")
        cs.execute("USE WAREHOUSE task_1_warehouse_mg")
        cs.execute("USE SCHEMA task_2_mg")
        ##cs.execute("SHOW VIEWS IN task_2_mg")
        ##print(cs.fetchall())
        ##runQueries(cs)
        cs.execute('SELECT * FROM "v_order_f" LIMIT 10')
        print(cs.fetchall())
    except Exception as e:
        print(f"Error: {e}") 
    finally:
        cs.close()
        ctx.close()
except Exception as e:
    print(f"Error: {e}")

[(609064, 1737763200000000000, 13, 'Drone', 'Electronics', 5, 925.69, 4628.450000000001, 4628.450000000001, 2025, 1, 1, 4, 41, 'Cynthia Dorsey MD', 'ohart@hotmail.com', 'Danielside', 'Australia'), (537537, 1737849600000000000, 14, 'Action Camera', 'Electronics', 4, 2048.76, 8195.04, 8195.04, 2025, 1, 1, 4, 67, 'Mrs. Jamie Smith', 'laurasilva@jones.com', 'South Nathanside', 'Germany'), (411957, 1738281600000000000, 9, 'External Hard Drive', 'Electronics', 3, 2110.69, 6332.07, 6332.07, 2025, 1, 1, 5, 11, 'Bonnie Phillips', 'pburke@bond-blair.com', 'East Diana', 'UK'), (183387, 1738368000000000000, 19, 'Portable Projector', 'Electronics', 1, 885.96, 885.96, 885.96, 2025, 1, 2, 5, 94, 'Jessica Burns', 'megan94@mueller.com', 'North Michaelhaven', 'Canada'), (307049, 1738454400000000000, 2, 'Smartphone', 'Electronics', 3, 2948.17, 8844.51, 8844.51, 2025, 1, 2, 5, 21, 'Christopher Steele', 'julielawson@gmail.com', 'Timland', 'Germany'), (531685, 1738540800000000000, 7, 'Monitor', 'Electronics

In [2]:
def runQueries(cs):    
    query = """
    CREATE OR REPLACE VIEW "v_order_f" AS
    SELECT 
        o."order_id",
        o."date",
        p."product_id",
        p."product_name",
        p."category",
        o."quantity",
        p."price",
        o."quantity" * p."price" AS "total_amount_local",

        -- Apply exchange rate only if price_currency is NOT 'USD'
        CASE 
            WHEN p."price_currency" = 'USD' THEN o."quantity" * p."price"
            ELSE o."quantity" * p."price" * ex."exchange_rate"
        END AS "total_amount_usd",
        
        -- Fiscal Attributes
        YEAR(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_year",
        QUARTER(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_quarter",
        MONTH(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_month",
        WEEK(TO_TIMESTAMP(o."date" / 1000000000)) AS "order_week",

        -- Customer Information
        cust."customer_id",
        cust."customer_name",
        cust."email",
        cust."customer_city",
        cust."country"

    FROM orders o
    JOIN products p ON o."product_id" = p."product_id"
    JOIN customers cust ON o."customer_id" = cust."customer_id"
    
    -- Join exchangerates based on price_currency (only if not USD)
    LEFT JOIN exchangerates ex 
        ON p."price_currency" = ex."source_currency" 
        AND ex."target_currency" = 'USD';
    """

    cs.execute(query)